# ALTO v1 — Multi-sample Context-aware Best-of-N

V1 mở rộng v0 theo hai trục độc lập: nhiều background trong mỗi context specification và nhiều latent seed trên mỗi background. Thiết kế này tách **within-sample variance** (seed) khỏi **between-sample variance** (background selection), đồng thời tách hard validity gate khỏi soft ranking score.

Luồng chính: **ADE20K candidate heap → stratified background selection → SDXL Best-of-16 → instance segmentation + pose + preservation scoring → variance/correlation/Best-of-N analysis**.


## Cấu trúc Drive

```text
/content/drive/MyDrive/Colab Notebooks/alto/v1/
├── Input/
│   ├── backgrounds/
│   ├── masks/
│   ├── semantic_masks/
│   └── dataset_manifest.csv
└── Output/
    ├── images/
    ├── candidate_previews/      # tùy chọn
    ├── contact_sheets/          # tùy chọn
    ├── candidate_audit.csv
    ├── ranking.csv
    ├── summary_by_sample.csv
    ├── summary_by_context.csv
    ├── variance_decomposition.csv
    ├── selection_score_correlations.csv
    ├── best_of_n_by_sample.csv
    ├── best_of_n_summary.csv
    └── experiment_manifest.json
```

Manual review được thực hiện riêng trong `alto_v1_manual_validation.ipynb`.


## 1. Mount Drive và cài dependencies

Colab đã có sẵn PyTorch, NumPy, pandas và SciPy; cell cài đặt chỉ bổ sung các thư viện model/dataset còn thiếu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q -U datasets diffusers transformers accelerate safetensors timm

In [ ]:
import gc
import hashlib
import heapq
import json
import math
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageChops, ImageDraw, ImageFilter
from IPython.display import display
from scipy import ndimage, stats
from datasets import load_dataset
from diffusers import AutoPipelineForInpainting
from transformers import (
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation,
    VitPoseForPoseEstimation,
)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not enabled")


## 2. Cấu hình thí nghiệm

Các ngưỡng chọn dữ liệu, kích thước mask, tham số sinh ảnh và model chấm điểm được tập trung trong `CONFIG`. Sau khi thay đổi rule chọn ảnh hoặc tạo mask, chạy một lần với `REBUILD_DATASET=True`; sau đó có thể đặt lại `False` để dùng cache trên Drive.

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/alto/v1")
INPUT_DIR = PROJECT_DIR / "Input"
OUTPUT_DIR = PROJECT_DIR / "Output"

BACKGROUND_DIR = INPUT_DIR / "backgrounds"
MASK_DIR = INPUT_DIR / "masks"
SEMANTIC_DIR = INPUT_DIR / "semantic_masks"
DATASET_MANIFEST_PATH = INPUT_DIR / "dataset_manifest.csv"
CANDIDATE_AUDIT_PATH = OUTPUT_DIR / "candidate_audit.csv"

# Bật True sau khi đổi rule chọn ảnh/mask; đặt False sau lần rebuild thành công.
REBUILD_DATASET = True
RUN_GENERATION = True
RUN_EVALUATION = True
# Preview/contact sheet không ảnh hưởng thí nghiệm; chỉ bật khi cần kiểm tra trực quan.
CREATE_PREVIEWS = False

for path in [INPUT_DIR, OUTPUT_DIR, BACKGROUND_DIR, MASK_DIR, SEMANTIC_DIR, OUTPUT_DIR / "images"]:
    path.mkdir(parents=True, exist_ok=True)
if CREATE_PREVIEWS:
    for path in [OUTPUT_DIR / "candidate_previews", OUTPUT_DIR / "contact_sheets"]:
        path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": "alto_v1_multi_sample",
    # Dataset streaming và candidate search.
    "dataset_id": "zhoubolei/scene_parse_150",
    "dataset_split": "validation",
    "dataset_seed": 42,
    "max_stream_examples": 10000,
    "shuffle_buffer_size": 500,
    "top_k_candidates_per_spec": 12,
    "min_candidates_before_early_stop": 12,
    "backgrounds_per_spec": 4,

    # Ngưỡng hình học khi chọn background và vị trí.
    "strict_min_foot_ground_ratio": 0.25,
    "strict_max_obstacle_ratio": 0.34,
    "strict_max_relative_distance": 0.40,
    "relaxed_min_foot_ground_ratio": 0.05,
    "relaxed_max_obstacle_ratio": 0.62,
    "relaxed_max_relative_distance": 0.55,

    # Kích thước và mức bảo vệ target mask.
    "standing_mask_width_ratio": 0.24,
    "standing_mask_height_ratio": 0.72,
    "relative_mask_width_ratio": 0.24,
    "relative_mask_height_ratio": 0.72,
    "sitting_mask_width_ratio": 0.30,
    "sitting_mask_height_ratio": 0.62,
    "shadow_extension_ratio": 0.04,
    "mask_corner_radius_ratio": 0.22,
    "reference_protection_dilation": 4,

    # Best-of-N generation.
    "num_seeds_per_sample": 16,
    "seed_start": 1000,
    "best_of_budgets": [1, 4, 8, 16],

    # Kích thước là giới hạn tối đa; resize vẫn giữ nguyên tỷ lệ ảnh.
    "model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    "width": 1024,
    "height": 1024,
    "num_inference_steps": 40,
    "guidance_scale": 8.0,
    "strength": 0.99,
    "mask_blur_radius": 16,

    # Automatic scorer.
    "scoring_version": "v1_pose_preservation_v2",
    "instance_segmentation_model_id": "facebook/mask2former-swin-small-coco-instance",
    "pose_model_id": "usyd-community/vitpose-base-simple",
    "person_detection_threshold": 0.35,
    "keypoint_threshold": 0.30,
    "hard_min_placement": 0.35,
    "hard_min_scale": 0.20,
    "hard_min_pose": 0.20,
    "hard_min_relation": 0.25,
    "hard_min_reference_preservation": 0.55,
    "hard_min_context_preservation": 0.55,
}

if CONFIG["min_candidates_before_early_stop"] > CONFIG["top_k_candidates_per_spec"]:
    raise ValueError("min_candidates_before_early_stop không được lớn hơn top_k_candidates_per_spec.")
if max(CONFIG["best_of_budgets"]) > CONFIG["num_seeds_per_sample"]:
    raise ValueError("Best-of-N budget không được lớn hơn num_seeds_per_sample.")

print(json.dumps(CONFIG, indent=2))

### Quyết định phạm vi v1

V1 tập trung vào thí nghiệm multi-background, variance decomposition, hard/soft evaluation, candidate-score correlation và Best-of-N. Các phần chưa đủ ổn định được tách khỏi baseline chính:

- Bỏ CLIP human-crop quality và prompt score; anatomy/photorealism được kiểm tra bằng manual review riêng.
- Hoãn adaptive-budget simulation cho đến khi automatic ranking được kiểm định tốt hơn.
- Manual validation nằm trong `alto_v1_manual_validation.ipynb` và chỉ đọc output của notebook này.
- Candidate preview, dataset cache preview và contact sheet đều được điều khiển bằng `CREATE_PREVIEWS`.

Thiết kế mặc định gồm 6 specification × 4 background × 16 seed = 384 candidate. Có thể chạy lại evaluation mà không tải diffusion model bằng `RUN_GENERATION=False`.


## 3. Nguồn ADE20K Parquet

Đọc trực tiếp validation Parquet ở chế độ streaming để không phải tải và giải nén toàn bộ SceneParse150 lên Drive.

In [ ]:
ADE20K_VALIDATION_PARQUET = (
    "https://huggingface.co/datasets/"
    "zhoubolei/scene_parse_150/resolve/"
    "refs%2Fconvert%2Fparquet/"
    "scene_parsing/validation/0000.parquet"
)

print("ADE20K validation source:")
print(ADE20K_VALIDATION_PARQUET)

## 4. Các lớp ADE20K cần dùng

SceneParse150 sử dụng dense semantic annotation. Notebook chỉ truy vấn một tập lớp nhỏ, không phân tích toàn bộ ontology.

Các ID dưới đây theo thứ tự chuẩn SceneParse150, với ảnh annotation thường lưu:

- `0`: unlabeled;
- `1`: wall;
- `3`: sky;
- `4`: floor;
- `7`: road;
- `10`: grass;
- `12`: sidewalk;
- `13`: person;
- `14`: earth;
- `16`: table;
- `20`: chair;
- `21`: car;
- `24`: sofa;
- `31`: armchair;
- `32`: seat;
- `53`: path;
- `70`: bench.

Các ID được kiểm tra trực tiếp ở bước chẩn đoán trước khi tìm candidate.

In [ ]:
ADE = {
    "wall": 1, "sky": 3, "floor": 4, "road": 7, "grass": 10,
    "sidewalk": 12, "person": 13, "earth": 14, "table": 16,
    "chair": 20, "car": 21, "sofa": 24, "armchair": 31,
    "seat": 32, "path": 53, "bench": 70,
}

INDOOR_GROUND = {ADE["floor"]}

OUTDOOR_GROUND = {
    ADE["road"], ADE["grass"], ADE["sidewalk"],
    ADE["earth"], ADE["path"],
}

ALL_GROUND = INDOOR_GROUND | OUTDOOR_GROUND
PERSON_LABEL = ADE["person"]

## 5. Định nghĩa context specifications

Sáu specification vẫn đại diện cho ba context của v0, nhưng mỗi specification sẽ chọn nhiều background từ heap. Mỗi specification có một `spec_id` và một scale prior dùng chung; sample thực nghiệm được định danh bằng `spec_id` và heap rank. Prompt được giữ cố định trong một specification để background và seed là hai nguồn biến thiên chính.


In [ ]:
SPECS = [
    {
        "spec_id": "spec_001",
        "context_type": "free_placement",
        "relation": "left_side",
        "reference_object": "",
        "reference_scale_range": None,
        "ground_labels": INDOOR_GROUND,
        "prompt": (
            "A photorealistic adult woman standing naturally in the empty space on the "
            "left side indoors. Complete body visible: head, torso, two arms, two legs, and "
            "both feet. Feet resting naturally on the floor. Natural anatomy and scene-matched "
            "scale, perspective, lighting, and shadows."
        ),
    },
    {
        "spec_id": "spec_002",
        "context_type": "free_placement",
        "relation": "center_foreground",
        "reference_object": "",
        "reference_scale_range": None,
        "ground_labels": OUTDOOR_GROUND,
        "prompt": (
            "A photorealistic adult man standing naturally in the exact center foreground "
            "outdoors. Complete body visible: head, torso, two arms, two legs, and both feet. "
            "Feet resting naturally on the ground. Natural anatomy and scene-matched scale, "
            "perspective, lighting, and shadows."
        ),
    },
    {
        "spec_id": "spec_003",
        "context_type": "object_relative",
        "relation": "beside",
        "reference_object": "table",
        "reference_scale_range": (1.2, 2.8),
        "object_labels": {ADE["table"]},
        "ground_labels": ALL_GROUND,
        "prompt": (
            "A photorealistic adult woman standing naturally beside the visible table. "
            "Complete body visible: head, torso, two arms, two legs, and both feet. Feet "
            "resting naturally on the floor. Natural anatomy and scene-matched scale, "
            "perspective, lighting, and shadows."
        ),
    },
    {
        "spec_id": "spec_004",
        "context_type": "object_relative",
        "relation": "near",
        "reference_object": "car",
        "reference_scale_range": (0.7, 1.7),
        "object_labels": {ADE["car"]},
        "ground_labels": OUTDOOR_GROUND,
        "prompt": (
            "A photorealistic adult man standing naturally beside the visible car. Complete "
            "body visible: head, torso, two arms, two legs, and both feet. Feet resting "
            "naturally on the road. Natural anatomy and scene-matched scale, perspective, "
            "lighting, and shadows."
        ),
    },
    {
        "spec_id": "spec_005",
        "context_type": "physical_interaction",
        "relation": "sitting_on",
        "reference_object": "bench_or_seat",
        "reference_scale_range": (0.7, 2.2),
        "object_labels": {ADE["bench"], ADE["chair"], ADE["seat"]},
        "prompt": (
            "A photorealistic adult woman sitting upright on the visible chair, bench, or "
            "seat. Hips on the seat, back toward any backrest, knees and feet toward the open "
            "side. Complete head, torso, two arms, two legs, and both feet visible. Natural "
            "anatomy, scale, perspective, lighting, and shadows."
        ),
    },
    {
        "spec_id": "spec_006",
        "context_type": "physical_interaction",
        "relation": "sitting_on",
        "reference_object": "sofa_or_armchair",
        "reference_scale_range": (0.5, 1.6),
        "object_labels": {ADE["sofa"], ADE["armchair"]},
        "prompt": (
            "A photorealistic adult man sitting upright on the visible sofa or armchair. "
            "Hips on the cushion, back toward the backrest, knees and feet toward the open "
            "side. Complete head, torso, two arms, two legs, and both feet visible. Natural "
            "anatomy, scale, perspective, lighting, and shadows."
        ),
    },
]

pd.DataFrame([
    {k: v for k, v in spec.items() if k not in {"ground_labels", "object_labels"}}
    for spec in SPECS
])


## 6. Đọc và chuẩn hóa dữ liệu ADE20K

Dataset streaming có thể trả ảnh dưới dạng PIL, path hoặc dictionary. Các hàm dưới đây dò field ảnh/mask theo mode, chuyển ảnh về RGB và kiểm tra semantic ID nằm trong khoảng `0..150`.

In [ ]:
def pil_from_value(value):
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, dict) and "path" in value:
        return Image.open(value["path"])
    if isinstance(value, str):
        return Image.open(value)
    raise TypeError(f"Không thể chuyển value thành PIL image: {type(value)}")

def detect_image_and_mask(example):
    """Dò field ảnh RGB và semantic mask từ một example của dataset."""
    image_candidates = []
    mask_candidates = []

    for key, value in example.items():
        try:
            pil = pil_from_value(value)
        except Exception:
            continue

        if pil.mode in {"L", "P", "I", "I;16"}:
            mask_candidates.append((key, pil))
        else:
            image_candidates.append((key, pil))

    if not image_candidates or not mask_candidates:
        raise ValueError(
            f"Không dò được image/mask. Fields: {list(example.keys())}"
        )

    # Ảnh RGB lớn nhất; mask grayscale lớn nhất.
    image_key, image = max(
        image_candidates,
        key=lambda item: item[1].width * item[1].height,
    )
    mask_key, mask = max(
        mask_candidates,
        key=lambda item: item[1].width * item[1].height,
    )

    return image_key, image.convert("RGB"), mask_key, mask

def normalize_semantic_mask(mask_image):
    """Chuyển mask về mảng int32 hai chiều và kiểm tra miền class ID."""
    arr = np.array(mask_image)

    if arr.ndim == 3:
        arr = arr[..., 0]

    arr = arr.astype(np.int32)

    if arr.min() < 0 or arr.max() > 150:
        raise ValueError(
            "Semantic mask có giá trị ngoài khoảng SceneParse150 "
            f"0..150: min={arr.min()}, max={arr.max()}"
        )

    return arr


## 7. Tạo stream và chuẩn hóa từng example

`prepare_stream()` tạo stream có shuffle xác định bởi seed. `process_example()` trả về `(image_key, image, mask_key, seg)` và resize semantic mask bằng nearest-neighbor nếu kích thước chưa khớp ảnh nguồn.

In [ ]:
def prepare_stream(seed_offset=0):
    """Tạo ADE20K stream có thứ tự shuffle tái lập được."""
    dataset = load_dataset(
        "parquet",
        data_files={"validation": ADE20K_VALIDATION_PARQUET},
        split="validation",
        streaming=True,
    )
    return dataset.shuffle(
        seed=CONFIG["dataset_seed"] + seed_offset,
        buffer_size=CONFIG["shuffle_buffer_size"],
    )

def process_example(example):
    """Trả về ảnh RGB và semantic mask cùng kích thước."""
    image_key, image, mask_key, mask_image = detect_image_and_mask(example)
    seg = normalize_semantic_mask(mask_image)

    if seg.shape != (image.height, image.width):
        seg = np.array(
            Image.fromarray(seg.astype(np.uint8)).resize(
                image.size, Image.Resampling.NEAREST
            )
        ).astype(np.int32)

    return image_key, image, mask_key, seg

## 8. Phân tích semantic layout

Candidate được đánh giá trực tiếp trên semantic mask, không dựa vào caption. Trước hết notebook xây dựng các phép toán dùng chung: connected component, bounding box, integral image và các chỉ số ground/obstacle/sky cho vùng người dự kiến.

In [ ]:
def class_mask(seg, labels):
    labels = list(labels)
    return np.isin(seg, labels)

def area_ratio(mask):
    return float(mask.mean())

def largest_component(binary):
    labeled, count = ndimage.label(binary)
    if count == 0:
        return None, 0

    sizes = ndimage.sum(
        binary,
        labeled,
        index=np.arange(1, count + 1),
    )
    best_label = int(np.argmax(sizes)) + 1
    component = labeled == best_label
    return component, int(sizes[best_label - 1])

def bbox_from_mask(binary):
    ys, xs = np.where(binary)
    if len(xs) == 0:
        return None
    return (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()) + 1,
        int(ys.max()) + 1,
    )

def touches_border(box, W, H, margin=0.025):
    x1, y1, x2, y2 = box
    return (
        x1 <= margin * W
        or y1 <= margin * H
        or x2 >= (1 - margin) * W
        or y2 >= (1 - margin) * H
    )

def integral_image(binary):
    return binary.astype(np.int32).cumsum(0).cumsum(1)

def rect_sum(integral, x1, y1, x2, y2):
    H, W = integral.shape
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    if x1 >= x2 or y1 >= y2:
        return 0

    total = integral[y2 - 1, x2 - 1]
    if x1 > 0:
        total -= integral[y2 - 1, x1 - 1]
    if y1 > 0:
        total -= integral[y1 - 1, x2 - 1]
    if x1 > 0 and y1 > 0:
        total += integral[y1 - 1, x1 - 1]
    return int(total)


### 8.1. Đánh giá vùng người

`make_person_box()` dựng box có kích thước tương đối theo ảnh và dịch box vào trong biên thay vì crop. `evaluate_person_box()` đo tiếp xúc chân–ground, tỷ lệ obstacle và tỷ lệ sky.

In [ ]:
def make_person_box(anchor_x, anchor_y, W, H, width_ratio=0.18, height_ratio=0.62):
    person_width = int(width_ratio * W)
    person_height = int(height_ratio * H)

    x1 = int(anchor_x - person_width / 2)
    x2 = x1 + person_width

    y2 = int(anchor_y)
    y1 = y2 - person_height

    # Dịch box vào trong ảnh thay vì crop trực tiếp nhằm giữ nguyên kích thước mask khi gần mép.
    if x1 < 0:
        x2 -= x1
        x1 = 0

    if x2 > W:
        shift = x2 - W
        x1 -= shift
        x2 = W

    if y1 < 0:
        y2 -= y1
        y1 = 0

    if y2 > H:
        shift = y2 - H
        y1 -= shift
        y2 = H

    return (
        max(0, int(x1)),
        max(0, int(y1)),
        min(W, int(x2)),
        min(H, int(y2)),
    )

def evaluate_person_box(seg, box, allowed_ground_labels, reference_mask=None):
    H, W = seg.shape
    x1, y1, x2, y2 = box
    area = max(1, (x2 - x1) * (y2 - y1))

    ground = class_mask(seg, allowed_ground_labels)
    ground_integral = integral_image(ground)

    # Dải chân dưới cùng của box phải tiếp xúc ground.
    foot_h = max(2, int(0.08 * (y2 - y1)))
    foot_ground = rect_sum(ground_integral, x1, y2 - foot_h, x2, y2) / max(1, (x2 - x1) * foot_h)

    allowed = ground.copy()
    allowed |= seg == 0

    if reference_mask is not None:
        allowed |= reference_mask

    obstacle = ~allowed
    obstacle_integral = integral_image(obstacle)

    obstacle_ratio = rect_sum(obstacle_integral, x1, y1, x2, y2) / area

    sky_ratio = float((seg[y1:y2, x1:x2] == ADE["sky"]).mean())

    return {
        "foot_ground_ratio": foot_ground,
        "obstacle_ratio": obstacle_ratio,
        "sky_ratio": sky_ratio,
    }


### 8.2. Free placement

Quét một lưới anchor thưa ở bên trái hoặc trung tâm ảnh. Candidate chỉ được giữ khi chân tiếp xúc ground, obstacle thấp và vùng phía trên không chứa quá nhiều sky. Với `center_foreground`, notebook ưu tiên cột khả dụng gần trục 50% nhất rồi mới so geometry score, tránh mask lệch trái chỉ vì ground score cao hơn một chút.

In [ ]:
def best_free_box(seg, relation, ground_labels):
    """Chọn box đứng tự do tốt nhất trên lưới anchor thưa."""
    H, W = seg.shape
    ground = class_mask(seg, ground_labels)

    if relation == "left_side":
        x_fracs = np.linspace(0.16, 0.38, 5)
        desired_x = 0.27
    else:
        x_fracs = np.linspace(0.40, 0.60, 5)
        desired_x = 0.50

    y_fracs = np.linspace(0.78, 0.96, 5)
    candidates = []

    for xf in x_fracs:
        for yf in y_fracs:
            x = int(xf * W)
            y = int(yf * H)

            if not ground[y, x]:
                continue

            box = make_person_box(x, y, W, H,
                                  width_ratio=CONFIG["standing_mask_width_ratio"],
                                  height_ratio=CONFIG["standing_mask_height_ratio"],
                                  )
            metrics = evaluate_person_box(seg, box, ground_labels)
            box_center_x = (box[0] + box[2]) / (2 * W)
            alignment = max(0.0, 1.0 - abs(box_center_x - desired_x) / 0.15)
            metrics["horizontal_alignment_score"] = alignment

            if metrics["foot_ground_ratio"] < 0.35:
                continue
            if metrics["obstacle_ratio"] > 0.32:
                continue
            if metrics["sky_ratio"] > 0.55:
                continue

            score = (
                1.6 * metrics["foot_ground_ratio"]
                + 1.2 * (1 - metrics["obstacle_ratio"])
                + 0.4 * (1 - metrics["sky_ratio"])
                + 0.3 * alignment
            )

            candidates.append((score, box, metrics, alignment))

    if not candidates:
        return None

    if relation == "center_foreground":
        # Ưu tiên cột gần trục giữa nhất; geometry score chỉ phá hòa trong cột đó.
        best_alignment = max(item[3] for item in candidates)
        candidates = [item for item in candidates if item[3] >= best_alignment - 1e-6]

    score, box, metrics, _ = max(candidates, key=lambda item: item[0])
    return score, box, metrics


### 8.3. Object-relative placement

Lấy connected component lớn nhất của reference object và thử anchor ở hai bên. Chế độ relaxed chấp nhận table sát biên, nhưng chỉ giữ phía còn đủ chỗ cho tâm người và giới hạn phần mask chồng lên table; nếu cả hai phía đều không hợp lệ thì ảnh vẫn bị loại.

In [ ]:
def best_relative_box(seg, object_labels, ground_labels, relaxed=False):
    """Chọn box đứng cạnh connected component lớn nhất của object."""
    H, W = seg.shape
    object_binary = class_mask(seg, object_labels)
    component, pixels = largest_component(object_binary)

    if component is None:
        return None

    component_ratio = pixels / (W * H)
    min_component_ratio = 0.004 if relaxed else 0.008
    if not (min_component_ratio <= component_ratio <= 0.35):
        return None

    object_box = bbox_from_mask(component)
    if object_box is None or (not relaxed and touches_border(object_box, W, H)):
        return None

    x1, y1, x2, y2 = object_box

    if relaxed:
        min_foot = CONFIG["relaxed_min_foot_ground_ratio"]
        max_obstacle = CONFIG["relaxed_max_obstacle_ratio"]
        max_distance = CONFIG["relaxed_max_relative_distance"]
        anchor_distances = [0.06, 0.10, 0.15, 0.21, 0.28]
        search_up_ratio = 0.24
        search_down_ratio = 0.32
    else:
        min_foot = CONFIG["strict_min_foot_ground_ratio"]
        max_obstacle = CONFIG["strict_max_obstacle_ratio"]
        max_distance = CONFIG["strict_max_relative_distance"]
        anchor_distances = [0.08, 0.14, 0.20, 0.27]
        search_up_ratio = 0.20
        search_down_ratio = 0.28

    anchors = []
    for distance_ratio in anchor_distances:
        anchors.extend([
            (max(0, x1 - int(distance_ratio * W)),
             min(H - 1, y2 + int(0.05 * H)), "left"),
            (min(W - 1, x2 + int(distance_ratio * W)),
             min(H - 1, y2 + int(0.05 * H)), "right"),
        ])

    ground = class_mask(seg, ground_labels)
    candidates = []

    for anchor_x, approximate_y, side in anchors:
        search_y1 = max(0, approximate_y - int(search_up_ratio * H))
        search_y2 = min(H, approximate_y + int(search_down_ratio * H))

        ground_rows = np.where(ground[search_y1:search_y2, anchor_x])[0]
        if len(ground_rows) == 0:
            continue

        anchor_y = search_y1 + int(ground_rows[-1])
        box = make_person_box(anchor_x, anchor_y, W, H,
                              width_ratio=CONFIG["relative_mask_width_ratio"],
                              height_ratio=CONFIG["relative_mask_height_ratio"],
                              )

        # Anchor bị clamp ở biên có thể đẩy box sang phía sai của object.
        person_center_x = (box[0] + box[2]) / 2
        wrong_side = (side == "left" and person_center_x >= x1) or (
            side == "right" and person_center_x <= x2
        )
        if wrong_side:
            continue

        metrics = evaluate_person_box(
            seg, box, ground_labels, reference_mask=component
        )
        person_area = max(1, (box[2] - box[0]) * (box[3] - box[1]))
        reference_overlap = rect_sum(
            integral_image(component), box[0], box[1], box[2], box[3]
        ) / person_area
        metrics["reference_overlap_ratio"] = reference_overlap

        if metrics["foot_ground_ratio"] < min_foot:
            continue
        if metrics["obstacle_ratio"] > max_obstacle:
            continue
        if reference_overlap > (0.15 if relaxed else 0.08):
            continue

        object_center_x = (x1 + x2) / 2
        normalized_distance = abs(person_center_x - object_center_x) / W

        if normalized_distance > max_distance:
            continue

        score = (
            1.4 * metrics["foot_ground_ratio"]
            + 1.0 * (1 - metrics["obstacle_ratio"])
            + 0.8 * component_ratio
            + 0.3 * (1 - normalized_distance)
            - 0.5 * reference_overlap
        )

        if not relaxed:
            score += 0.15

        candidates.append(
            (score, box, object_box, side, metrics, component_ratio)
        )

    if not candidates:
        return None

    return max(candidates, key=lambda item: item[0])


### 8.4. Physical interaction

Với hành động ngồi, target box được neo gần mặt ghế và chồng một phần lên reference object để mô hình có đủ không gian tạo thân, hông và chân. Strict search ưu tiên ghế đủ lớn, không sát biên; relaxed search chỉ nới các ngưỡng hình học khi chưa tìm đủ candidate.

In [ ]:
def best_interaction_box(seg, object_labels, relaxed=False):
    """Đặt box ngồi chồng có kiểm soát lên reference object."""
    H, W = seg.shape
    object_binary = class_mask(seg, object_labels)
    component, pixels = largest_component(object_binary)

    if component is None:
        return None

    component_ratio = pixels / (W * H)
    min_component_ratio = 0.005 if relaxed else 0.010
    if not (min_component_ratio <= component_ratio <= 0.35):
        return None

    object_box = bbox_from_mask(component)
    if not relaxed and touches_border(object_box, W, H):
        return None

    x1, y1, x2, y2 = object_box
    object_w = x2 - x1
    object_h = y2 - y1

    min_width = (0.06 if relaxed else 0.10) * W
    min_height = (0.04 if relaxed else 0.06) * H
    if object_w < min_width or object_h < min_height:
        return None

    anchor_x = int((x1 + x2) / 2)
    # Neo gần mặt ghế để box bao phủ thân trên và phần chân phía trước ghế.
    seat_y = int(y1 + 0.70 * object_h)
    anchor_y = min(H - 1, max(seat_y + int(0.06 * H), y2 + int(0.10 * H)))

    adaptive_width_ratio = max(
        CONFIG["sitting_mask_width_ratio"],
        min(0.32, object_w / W * 1.05),
    )

    box = make_person_box(
        anchor_x,
        anchor_y,
        W,
        H,
        width_ratio=adaptive_width_ratio,
        height_ratio=CONFIG["sitting_mask_height_ratio"],
    )

    metrics = evaluate_person_box(seg, box, ALL_GROUND, reference_mask=component)

    overlap = rect_sum(
        integral_image(component), box[0], box[1], box[2], box[3]
    ) / max(1, (box[2] - box[0]) * (box[3] - box[1]))

    if overlap < (0.004 if relaxed else 0.008):
        return None
    if metrics["obstacle_ratio"] > (0.62 if relaxed else 0.52):
        return None

    score = (1.2 * component_ratio + 1.0 * overlap + 0.7 * (1 - metrics["obstacle_ratio"]))

    return (
        score,
        box,
        object_box,
        "overlap",
        metrics,
        component_ratio,
    )


### 8.5. General person-canvas mask

Mọi context dùng cùng một person-canvas mask bo góc; mask không mã hóa trước pose đứng/ngồi. Với object-relative, reference object được bảo vệ. Physical interaction cho phép canvas chồng lên ghế vì vùng cơ thể che khuất bắt buộc phải thay đổi.


In [ ]:
def add_shadow_space(box, H, extension_ratio=None):
    """Chừa một dải nhỏ dưới chân để mô hình có thể tạo contact shadow."""
    if extension_ratio is None:
        extension_ratio = CONFIG["shadow_extension_ratio"]
    x1, y1, x2, y2 = box
    extension = int(extension_ratio * max(1, y2 - y1))
    return x1, y1, x2, min(H, y2 + extension)


def build_person_canvas_mask(image_size, box):
    """Tạo person-canvas bo góc liên tục, dùng chung cho mọi pose."""
    W, H = image_size
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    mask = Image.new("L", (W, H), 0)
    radius = min(w // 2, int(CONFIG["mask_corner_radius_ratio"] * min(w, h)))
    ImageDraw.Draw(mask).rounded_rectangle(
        (x1, y1, x2 - 1, y2 - 1), radius=radius, fill=255
    )
    return np.asarray(mask) > 0


def build_target_mask(seg, target_box, spec):
    """Tạo hard mask liên tục và giữ nguyên reference object khi không có occlusion."""
    H, W = seg.shape
    target = build_person_canvas_mask((W, H), target_box)

    if spec["context_type"] == "physical_interaction":
        # Interaction cần cho phép cơ thể che khuất một phần ghế trong canvas.
        return target.astype(np.uint8) * 255

    x1, _, x2, y2 = target_box
    _, _, _, shadow_y2 = add_shadow_space(target_box, H)
    center_x = (x1 + x2) // 2
    shadow_half_w = max(1, int(0.30 * (x2 - x1)))
    target[y2:shadow_y2, center_x-shadow_half_w:center_x+shadow_half_w] = True

    object_labels = spec.get("object_labels")
    if object_labels:
        reference = class_mask(seg, object_labels)
        reference, _ = largest_component(reference)
        if reference is not None:
            protected = ndimage.binary_dilation(
                reference, iterations=CONFIG["reference_protection_dilation"]
            )
            target &= ~protected

    return target.astype(np.uint8) * 255


## 9. Tìm candidate trên stream

Mỗi ảnh được đối chiếu với sáu specification. Min-heap chỉ giữ top-`K`, tránh lưu toàn bộ stream trong RAM. Một workflow duy nhất chạy strict search trước, sau đó tự động dùng relaxed supplement cho các specification còn thiếu candidate. `max_stream_examples` là giới hạn riêng cho mỗi lượt stream.


In [ ]:
def push_top_k(heap, item, k):
    if len(heap) < k:
        heapq.heappush(heap, item)
    else:
        heapq.heappushpop(heap, item)

def spec_match(spec, seg, relaxed=False):
    """Đánh giá một semantic mask theo đúng strategy của sample spec."""
    if (seg == PERSON_LABEL).any():
        return None

    H, W = seg.shape
    aspect = W / H

    if relaxed:
        if W < 400 or H < 300 or not (0.90 <= aspect <= 2.50):
            return None
    else:
        if W < 480 or H < 360 or not (1.05 <= aspect <= 2.20):
            return None

    if spec["context_type"] == "free_placement":
        ground_ratio = area_ratio(class_mask(seg, spec["ground_labels"]))
        minimum_ground_ratio = 0.07 if relaxed else 0.10

        if ground_ratio < minimum_ground_ratio:
            return None

        result = best_free_box(seg, spec["relation"], spec["ground_labels"])
        if result is None:
            return None

        score, target_box, metrics = result
        return {
            "score": score + 0.5 * ground_ratio,
            "target_box": target_box,
            "reference_box": None,
            "placement_side": spec["relation"],
            "metrics": metrics,
            "semantic_area_ratio": ground_ratio,
            "selection_mode": "relaxed" if relaxed else "strict",
        }

    if spec["context_type"] == "object_relative":
        result = best_relative_box(
            seg, spec["object_labels"], spec["ground_labels"], relaxed=relaxed
        )
        if result is None:
            return None

        score, target_box, reference_box, side, metrics, ratio = result
        return {
            "score": score,
            "target_box": target_box,
            "reference_box": reference_box,
            "placement_side": side,
            "metrics": metrics,
            "semantic_area_ratio": ratio,
            "selection_mode": "relaxed" if relaxed else "strict",
        }

    result = best_interaction_box(seg, spec["object_labels"], relaxed=relaxed)
    if result is None:
        return None

    score, target_box, reference_box, side, metrics, ratio = result
    return {
        "score": score,
        "target_box": target_box,
        "reference_box": reference_box,
        "placement_side": side,
        "metrics": metrics,
        "semantic_area_ratio": ratio,
        "selection_mode": "relaxed" if relaxed else "strict",
    }


### 9.1. Strict search với relaxed fallback

`run_candidate_search()` quản lý toàn bộ hai giai đoạn và trả về heap đã hoàn chỉnh. Strict và relaxed vẫn được ghi trong `selection_mode` để audit, nhưng không còn hai block thực thi rời nhau. Fingerprint ngăn cùng một ảnh bị thêm lại khi lượt relaxed dùng shuffle seed khác.


In [ ]:
def collect_candidates(heaps, specs, relaxed, seed_offset, counter, target_count):
    stream = prepare_stream(seed_offset=seed_offset)
    processed = 0
    start = time.time()
    mode = "relaxed" if relaxed else "strict"

    for example_index, example in enumerate(stream):
        if processed >= CONFIG["max_stream_examples"]:
            break

        _, image, _, seg = process_example(example)
        fingerprint = None

        for spec in specs:
            spec_id = spec["spec_id"]
            if len(heaps[spec_id]) >= target_count:
                continue

            match = spec_match(spec, seg, relaxed=relaxed)
            if match is None:
                continue

            if fingerprint is None:
                fingerprint = hashlib.blake2b(
                    image.tobytes(), digest_size=8
                ).hexdigest()
            known = {item[2]["fingerprint"] for item in heaps[spec_id]}
            if fingerprint in known:
                continue

            payload = {
                "example_index": example_index,
                "fingerprint": fingerprint,
                "image": image.copy(),
                "semantic_mask": seg.copy(),
                **match,
            }
            push_top_k(
                heaps[spec_id],
                (match["score"], counter, payload),
                CONFIG["top_k_candidates_per_spec"],
            )
            counter += 1

        processed += 1
        if processed % 100 == 0:
            counts = {spec["spec_id"]: len(heaps[spec["spec_id"]]) for spec in specs}
            print(f"{mode} processed={processed} candidates={counts} elapsed={time.time()-start:.1f}s")

        if all(len(heaps[spec["spec_id"]]) >= target_count for spec in specs):
            break

    return processed, counter


def run_candidate_search(specs):
    target_count = CONFIG["min_candidates_before_early_stop"]
    heaps = {spec["spec_id"]: [] for spec in specs}

    strict_processed, counter = collect_candidates(
        heaps, specs, relaxed=False, seed_offset=0,
        counter=0, target_count=target_count,
    )
    print("Strict processed examples:", strict_processed)

    underfilled = [
        spec for spec in specs
        if len(heaps[spec["spec_id"]]) < target_count
    ]
    if underfilled:
        print("Running relaxed supplement for:", [spec["spec_id"] for spec in underfilled])
        relaxed_processed, _ = collect_candidates(
            heaps, underfilled, relaxed=True, seed_offset=1,
            counter=counter, target_count=target_count,
        )
        print("Relaxed processed examples:", relaxed_processed)

    final_counts = {spec_id: len(heap) for spec_id, heap in heaps.items()}
    print("Final candidate counts:", final_counts)

    shortfall = {
        spec_id: count for spec_id, count in final_counts.items()
        if count < target_count
    }
    if shortfall:
        print("WARNING - candidate shortfall:", shortfall)

    minimum = CONFIG["backgrounds_per_spec"]
    insufficient = {
        spec_id: count for spec_id, count in final_counts.items()
        if count < minimum
    }
    if insufficient:
        raise RuntimeError(
            f"Không đủ {minimum} background cho mỗi specification: {insufficient}"
        )
    return heaps


if REBUILD_DATASET:
    top_heaps = run_candidate_search(SPECS)


## 10. Chọn nhiều background từ candidate heap

V1 không chỉ lấy rank đầu. Mỗi specification chọn bốn rank trải đều trên heap để tạo biến thiên candidate-selection score. Rank vẫn bắt đầu từ 1. Nếu heap không đủ số background tối thiểu, notebook dừng thay vì lặp lại cùng một ảnh.


In [ ]:
def stratified_heap_ranks(heap_size, count):
    if heap_size < count:
        raise RuntimeError(f"Heap chỉ có {heap_size} candidate, cần ít nhất {count}.")
    ranks = np.rint(np.linspace(1, heap_size, count)).astype(int).tolist()
    return sorted(set(ranks))


def preview_candidates(heap, spec_id, cols=4, thumb=256):
    ranked = sorted(heap, key=lambda item: item[0], reverse=True)
    selected_ranks = stratified_heap_ranks(
        len(ranked), CONFIG["backgrounds_per_spec"]
    )
    label_h = 44
    rows = math.ceil(len(ranked) / cols)
    sheet = Image.new("RGB", (cols * thumb, rows * (thumb + label_h)), "white")
    draw_sheet = ImageDraw.Draw(sheet)

    for index, (score, _, payload) in enumerate(ranked):
        rank = index + 1
        image = payload["image"].copy()
        draw = ImageDraw.Draw(image)
        draw.rectangle(payload["target_box"], outline="red", width=max(2, image.width // 250))
        if payload["reference_box"] is not None:
            draw.rectangle(payload["reference_box"], outline="lime", width=max(2, image.width // 250))
        image.thumbnail((thumb, thumb))

        x = (index % cols) * thumb
        y = (index // cols) * (thumb + label_h)
        canvas = Image.new("RGB", (thumb, thumb), "white")
        canvas.paste(image, ((thumb - image.width) // 2, (thumb - image.height) // 2))
        sheet.paste(canvas, (x, y))
        marker = "SELECTED" if rank in selected_ranks else ""
        draw_sheet.text((x + 6, y + thumb + 4), f"rank={rank} score={score:.3f} {marker}", fill="black")
        draw_sheet.text(
            (x + 6, y + thumb + 22),
            f"idx={payload['example_index']} {payload['selection_mode']}",
            fill="black",
        )

    preview_path = OUTPUT_DIR / "candidate_previews" / f"{spec_id}.png"
    sheet.save(preview_path)
    print(spec_id, "| selected ranks:", selected_ranks, "|", preview_path)
    display(sheet)


if REBUILD_DATASET and CREATE_PREVIEWS:
    for spec in SPECS:
        preview_candidates(top_heaps[spec["spec_id"]], spec["spec_id"])


### 10.1. Lưu multi-sample dataset cache

`candidate_audit.csv` lưu toàn bộ heap; `dataset_manifest.csv` lưu bốn background đã chọn cho mỗi specification. `selection_score` và `heap_rank` được truyền xuyên suốt đến bảng score để phân tích tương quan sau generation.


In [ ]:
if REBUILD_DATASET:
    selected_rows = []
    audit_rows = []

    for spec in SPECS:
        spec_id = spec["spec_id"]
        ranked = sorted(top_heaps[spec_id], key=lambda item: item[0], reverse=True)
        selected_ranks = stratified_heap_ranks(
            len(ranked), CONFIG["backgrounds_per_spec"]
        )

        for rank, (score, _, payload) in enumerate(ranked, start=1):
            audit_rows.append({
                "spec_id": spec_id,
                "heap_rank": rank,
                "selected": int(rank in selected_ranks),
                "selection_score": score,
                "example_index": payload["example_index"],
                "context_type": spec["context_type"],
                "relation": spec["relation"],
                "reference_object": spec["reference_object"],
                "target_box": json.dumps(payload["target_box"]),
                "reference_box": json.dumps(payload["reference_box"]),
                "placement_side": payload["placement_side"],
                "semantic_area_ratio": payload["semantic_area_ratio"],
                "selection_mode": payload.get("selection_mode", "strict"),
                **payload["metrics"],
            })

        for heap_rank in selected_ranks:
            selection_score, _, selected = ranked[heap_rank - 1]
            sample_id = f"{spec_id}_r{heap_rank:02d}"
            image_path = BACKGROUND_DIR / f"{sample_id}.jpg"
            semantic_path = SEMANTIC_DIR / f"{sample_id}.png"
            mask_path = MASK_DIR / f"{sample_id}.png"

            selected["image"].save(image_path, quality=95)
            seg = selected["semantic_mask"].astype(np.uint8)
            Image.fromarray(seg).save(semantic_path)
            target_mask = Image.fromarray(
                build_target_mask(selected["semantic_mask"], selected["target_box"], spec)
            )
            target_mask.save(mask_path)

            selected_rows.append({
                "sample_id": sample_id,
                "spec_id": spec_id,
                "heap_rank": heap_rank,
                "selection_score": selection_score,
                "source_dataset": CONFIG["dataset_id"],
                "source_split": CONFIG["dataset_split"],
                "source_example_index": selected["example_index"],
                "context_type": spec["context_type"],
                "relation": spec["relation"],
                "reference_object": spec["reference_object"],
                "object_label_ids": json.dumps(sorted(spec.get("object_labels", []))),
                "reference_scale_range": json.dumps(spec["reference_scale_range"]),
                "reference_box": json.dumps(selected["reference_box"]),
                "target_box": json.dumps(selected["target_box"]),
                "background_path": str(image_path.relative_to(INPUT_DIR)),
                "semantic_mask_path": str(semantic_path.relative_to(INPUT_DIR)),
                "mask_path": str(mask_path.relative_to(INPUT_DIR)),
                "prompt": spec["prompt"],
            })

    dataset_df = pd.DataFrame(selected_rows)
    audit_df = pd.DataFrame(audit_rows)
    dataset_df.to_csv(DATASET_MANIFEST_PATH, index=False, sep=";", encoding="utf-8-sig")
    audit_df.to_csv(CANDIDATE_AUDIT_PATH, index=False, sep=";", encoding="utf-8-sig")
    print("Samples:", len(dataset_df), "| per spec:", dataset_df.groupby("spec_id").size().to_dict())
else:
    if not DATASET_MANIFEST_PATH.exists():
        raise FileNotFoundError("REBUILD_DATASET=False nhưng chưa có dataset_manifest.csv")
    dataset_df = pd.read_csv(DATASET_MANIFEST_PATH, sep=";", encoding="utf-8-sig")

display(dataset_df.head())


## 11. Kiểm tra trực quan dataset cache (tùy chọn)

Khi `CREATE_PREVIEWS=True`, cell hiển thị background, semantic annotation và target mask cạnh nhau để kiểm tra vùng chân/ghế và reference object trước khi chạy SDXL. Cell được bỏ qua hoàn toàn ở cấu hình mặc định.


In [ ]:
if CREATE_PREVIEWS:
    for row in dataset_df.itertuples(index=False):
        background = Image.open(INPUT_DIR / row.background_path).convert("RGB")
        semantic = Image.open(INPUT_DIR / row.semantic_mask_path).convert("L")
        target = Image.open(INPUT_DIR / row.mask_path).convert("L")

        size = (300, 300)

        bg_thumb = background.copy()
        bg_thumb.thumbnail(size)

        sem_thumb = semantic.convert("RGB")
        sem_thumb.thumbnail(size)

        target_thumb = target.convert("RGB")
        target_thumb.thumbnail(size)

        canvas = Image.new("RGB", (900, 350), "white")
        canvas.paste(bg_thumb, (0, 0))
        canvas.paste(sem_thumb, (300, 0))
        canvas.paste(target_thumb, (600, 0))

        draw = ImageDraw.Draw(canvas)
        draw.text(
            (10, 315),
            f"{row.sample_id} | {row.context_type} | "
            f"{row.reference_object}",
            fill="black",
        )

        display(canvas)
else:
    print("CREATE_PREVIEWS=False: bỏ qua dataset cache preview.")


## 12. Khởi tạo SDXL Inpainting

Generation yêu cầu GPU. Model dùng FP16 và CPU offload để giảm VRAM; negative prompt tập trung loại lỗi anatomy, nhiều người và biến dạng background.

In [ ]:
NEGATIVE_PROMPT = (
    "empty scene, cropped body, headless, faceless, missing limbs, extra limbs, "
    "fused limbs, multiple people, floating body, malformed anatomy, malformed face, "
    "malformed hands, malformed feet, wrong scale, wrong perspective, ghost silhouette, "
    "rectangular seam, distorted furniture, blurry, low quality"
)

if RUN_GENERATION:
    assert torch.cuda.is_available(), "Generation cần GPU."
    torch.cuda.empty_cache()
    gc.collect()
    pipe = AutoPipelineForInpainting.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.enable_model_cpu_offload()
    pipe.set_progress_bar_config(disable=False)

    tokenizers = [t for t in (pipe.tokenizer, pipe.tokenizer_2) if t is not None]
    texts = [spec["prompt"] for spec in SPECS] + [NEGATIVE_PROMPT]
    for text in texts:
        counts = [len(tokenizer(text, truncation=False).input_ids) for tokenizer in tokenizers]
        limits = [tokenizer.model_max_length for tokenizer in tokenizers]
        if any(count > limit for count, limit in zip(counts, limits)):
            raise ValueError(f"Prompt vượt CLIP token limit: counts={counts}, limits={limits}: {text}")
    print("Loaded:", CONFIG["model_id"])
else:
    print("RUN_GENERATION=False: bỏ qua diffusion model.")


## 13. Chuẩn bị input và sinh ảnh

Ảnh và mask được resize đồng tỷ lệ về bội số của 8. Mọi sample dùng chung feathered mask trên toàn ảnh, không crop/zoom riêng vùng mask. Bước composite cuối vẫn bị giới hạn bởi hard mask, nên pixel bên ngoài vùng cho phép được lấy nguyên vẹn từ background.

In [ ]:
def load_inpainting_inputs(row):
    """Load ảnh/mask và tạo soft mask ở kích thước hợp lệ cho SDXL."""
    background = Image.open(INPUT_DIR / row["background_path"]).convert("RGB")
    mask = Image.open(INPUT_DIR / row["mask_path"]).convert("L")
    scale = min(CONFIG["width"] / background.width, CONFIG["height"] / background.height)
    size = tuple(max(8, int(d * scale) // 8 * 8) for d in background.size)
    background = background.resize(size, Image.Resampling.LANCZOS)
    mask = mask.resize(size, Image.Resampling.NEAREST)
    blurred_mask = mask.filter(ImageFilter.GaussianBlur(CONFIG["mask_blur_radius"]))
    soft_mask = ImageChops.darker(mask, blurred_mask)
    return background, soft_mask

def generate_image(prompt, background, soft_mask, seed):
    """Sinh một candidate; cùng input và seed sẽ cho kết quả tái lập."""
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    generated = pipe(
        prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
        image=background, mask_image=soft_mask, generator=generator,
        num_inference_steps=CONFIG["num_inference_steps"],
        guidance_scale=CONFIG["guidance_scale"], strength=CONFIG["strength"],
        width=background.width, height=background.height,
    ).images[0]
    return Image.composite(generated, background, soft_mask)

## 14. Sinh tối đa 16 seed cho mỗi background

Mọi sample dùng cùng danh sách seed, tạo thiết kế cân bằng cho variance decomposition. `RUN_GENERATION=False` cho phép dùng lại `runs.csv`. Best-of-1/4/8/16 được tính từ các prefix seed lồng nhau, không chạy model bốn lần.


In [ ]:
seeds = list(range(
    CONFIG["seed_start"],
    CONFIG["seed_start"] + CONFIG["num_seeds_per_sample"],
))
runs_path = OUTPUT_DIR / "runs.csv"

if RUN_GENERATION:
    records = []
    for _, row in dataset_df.iterrows():
        sample_id = row["sample_id"]
        sample_dir = OUTPUT_DIR / "images" / sample_id
        sample_dir.mkdir(parents=True, exist_ok=True)
        background, soft_mask = load_inpainting_inputs(row)

        for seed in seeds:
            output_path = sample_dir / f"seed_{seed}.png"
            record = {
                "sample_id": sample_id,
                "spec_id": row["spec_id"],
                "heap_rank": row["heap_rank"],
                "selection_score": row["selection_score"],
                "context_type": row["context_type"],
                "seed": seed,
                "output_path": str(output_path),
                "status": "started",
                "elapsed_seconds": None,
                "error": "",
            }
            try:
                start = time.time()
                result = generate_image(row["prompt"], background, soft_mask, seed)
                result.save(output_path)
                record["elapsed_seconds"] = time.time() - start
                record["status"] = "success"
                print(sample_id, seed, f'{record["elapsed_seconds"]:.1f}s')
            except Exception as exc:
                record["status"] = "failed"
                record["error"] = repr(exc)
                print("FAILED", sample_id, seed, exc)
            records.append(record)

    runs_df = pd.DataFrame(records)
    runs_df.to_csv(runs_path, index=False, sep=";", encoding="utf-8-sig")
else:
    if not runs_path.exists():
        raise FileNotFoundError("RUN_GENERATION=False nhưng chưa có runs.csv")
    runs_df = pd.read_csv(runs_path, sep=";", encoding="utf-8-sig")

display(runs_df.groupby(["spec_id", "status"]).size().rename("count").reset_index())


## 15. Tạo contact sheet (tùy chọn)

Khi `CREATE_PREVIEWS=True`, các output thành công được ghép theo sample và ghi seed dưới mỗi ảnh. Cell được bỏ qua hoàn toàn ở cấu hình mặc định.


In [ ]:
def make_contact_sheet(group, sample_id, cols=4, thumb=256):
    group = group[group["status"] == "success"]
    if group.empty:
        return None

    label_h = 34
    rows = math.ceil(len(group) / cols)

    sheet = Image.new("RGB", (cols * thumb, rows * (thumb + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for index, row in enumerate(group.itertuples(index=False)):
        image = Image.open(row.output_path).convert("RGB")
        image.thumbnail((thumb, thumb))

        x = (index % cols) * thumb
        y = (index // cols) * (thumb + label_h)

        cell = Image.new("RGB", (thumb, thumb), "white")
        cell.paste(image, ((thumb - image.width) // 2, (thumb - image.height) // 2))
        sheet.paste(cell, (x, y))
        draw.text((x + 8, y + thumb + 8), f"seed={row.seed}", fill="black")

    path = OUTPUT_DIR / "contact_sheets" / f"{sample_id}.png"
    sheet.save(path)
    return path


if CREATE_PREVIEWS:
    contact_sheet_paths = []
    for sample_id, group in runs_df.groupby("sample_id"):
        path = make_contact_sheet(group, sample_id)
        if path:
            contact_sheet_paths.append(path)
    print(f"Saved {len(contact_sheet_paths)} contact sheets.")
    for path in contact_sheet_paths[:3]:
        display(Image.open(path))
else:
    print("CREATE_PREVIEWS=False: bỏ qua contact sheets.")


## 16. Evaluation

Person instance segmentation là gate và cung cấp person mask; detection confidence chỉ áp dụng ở ngưỡng detection và được lưu để audit, không tham gia chọn target hay soft score. Target person được chọn theo khoảng cách hình học tới target box. ViTPose đánh giá keypoint coverage, tư thế đứng/ngồi và contact. Các pixel trong edit mask được chia thành person, reference object còn nhìn thấy và context còn lại để đánh giá preservation đúng vùng.

Anatomy và photorealism được kiểm tra ở manual validation notebook riêng.


In [ ]:
if RUN_EVALUATION:
    assert torch.cuda.is_available(), "Evaluation cần GPU."
    if "pipe" in globals():
        del pipe
    torch.cuda.empty_cache()
    gc.collect()

    DEVICE = "cuda"
    instance_processor = AutoImageProcessor.from_pretrained(
        CONFIG["instance_segmentation_model_id"]
    )
    instance_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        CONFIG["instance_segmentation_model_id"]
    ).to(DEVICE).eval()

    pose_processor = AutoImageProcessor.from_pretrained(CONFIG["pose_model_id"])
    pose_model = VitPoseForPoseEstimation.from_pretrained(
        CONFIG["pose_model_id"]
    ).to(DEVICE).eval()

    print("Evaluator models loaded.")


### 16.1. Geometry, masks và scale prior

Scale score kết hợp target-region fit với khoảng tỷ lệ người/reference object theo loại vật thể. Đây là prior mềm theo phối cảnh cục bộ, tốt hơn chỉ so người với mask nhưng vẫn cần manual validation.


In [ ]:
def parse_json(value, default=None):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if isinstance(value, (list, tuple, dict)):
        return value
    return json.loads(value)


def resize_box(box, old_size, new_size):
    if box is None:
        return None
    sx, sy = new_size[0] / old_size[0], new_size[1] / old_size[1]
    x1, y1, x2, y2 = box
    return [x1 * sx, y1 * sy, x2 * sx, y2 * sy]


def box_area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def box_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    return intersection / max(1e-6, box_area(a) + box_area(b) - intersection)


def center_score(person_box, target_box):
    px = (person_box[0] + person_box[2]) / 2
    py = (person_box[1] + person_box[3]) / 2
    tx = (target_box[0] + target_box[2]) / 2
    ty = (target_box[1] + target_box[3]) / 2
    diagonal = math.hypot(target_box[2] - target_box[0], target_box[3] - target_box[1])
    return float(np.exp(-2.0 * math.hypot(px - tx, py - ty) / max(1.0, diagonal)))


def placement_score(person_box, target_box):
    return float(np.clip(0.55 * center_score(person_box, target_box) + 0.45 * box_iou(person_box, target_box), 0, 1))


def interval_score(value, bounds, sigma=0.40):
    if bounds is None:
        return np.nan
    low, high = bounds
    if low <= value <= high:
        return 1.0
    boundary = low if value < low else high
    return float(np.exp(-abs(math.log(max(value, 1e-6) / boundary)) / sigma))


def scale_score(person_box, target_box, reference_box, reference_range):
    person_h = max(1.0, person_box[3] - person_box[1])
    target_h = max(1.0, target_box[3] - target_box[1])
    target_fit = float(np.exp(-abs(math.log(person_h / target_h)) / 0.45))
    if reference_box is None or reference_range is None:
        return target_fit, np.nan
    reference_h = max(1.0, reference_box[3] - reference_box[1])
    reference_fit = interval_score(person_h / reference_h, reference_range)
    return 0.40 * target_fit + 0.60 * reference_fit, reference_fit


def preservation_score(original, generated, region, scale=0.08):
    if int(region.sum()) < 32:
        return np.nan
    a = np.asarray(original, dtype=np.float32) / 255.0
    b = np.asarray(generated, dtype=np.float32) / 255.0
    mae = float(np.abs(a - b)[region].mean())
    return float(np.exp(-mae / scale))


### 16.2. Person mask và pose

Mask2Former thực hiện instance segmentation; person instance gần tâm target box nhất được chọn. ViTPose chạy top-down trên person box để đánh giá keypoint coverage và tư thế. Với physical interaction, contact chỉ đóng góp vào relation score để tránh bị tính hai lần trong pose score.


In [ ]:
@torch.inference_mode()
def segment_people(image):
    inputs = instance_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = instance_model(**inputs)
    result = instance_processor.post_process_instance_segmentation(
        outputs,
        threshold=CONFIG["person_detection_threshold"],
        target_sizes=[(image.height, image.width)],
        return_binary_maps=True,
    )[0]
    maps = result.get("segmentation")
    if maps is None:
        return []
    maps = maps.detach().cpu().numpy()
    if maps.ndim == 2:
        maps = maps[None]
    people = []
    for index, info in enumerate(result["segments_info"]):
        label = instance_model.config.id2label.get(info["label_id"], "").lower()
        if label != "person" or index >= len(maps):
            continue
        mask = maps[index].astype(bool)
        box = bbox_from_mask(mask)
        if box is not None:
            people.append({"score": float(info["score"]), "mask": mask, "box": box})
    return people


def select_target_person(people, target_box):
    if not people:
        return None
    return max(people, key=lambda person: center_score(person["box"], target_box))


@torch.inference_mode()
def estimate_pose(image, person_box):
    x1, y1, x2, y2 = person_box
    boxes = np.asarray([[x1, y1, x2 - x1, y2 - y1]], dtype=np.float32)
    inputs = pose_processor(images=image, boxes=[boxes], return_tensors="pt").to(DEVICE)
    outputs = pose_model(**inputs)
    results = pose_processor.post_process_pose_estimation(
        outputs,
        boxes=[boxes],
        target_sizes=[(image.height, image.width)],
    )
    if not results or not results[0]:
        return None, None
    pose = results[0][0]
    return pose["keypoints"].detach().cpu().numpy(), pose["scores"].detach().cpu().numpy()


def angle_degrees(a, b, c):
    u, v = np.asarray(a) - np.asarray(b), np.asarray(c) - np.asarray(b)
    denominator = np.linalg.norm(u) * np.linalg.norm(v)
    if denominator < 1e-6:
        return np.nan
    cosine = np.clip(np.dot(u, v) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def pose_metrics(keypoints, scores, context_type, reference_box, target_box):
    if keypoints is None:
        return 0.0, 0.0, 0.0
    visible = scores >= CONFIG["keypoint_threshold"]
    coverage = float(visible.mean())
    leg_angles = []
    for hip, knee, ankle in [(11, 13, 15), (12, 14, 16)]:
        if visible[[hip, knee, ankle]].all():
            leg_angles.append(angle_degrees(keypoints[hip], keypoints[knee], keypoints[ankle]))

    if context_type == "physical_interaction":
        angle_score = np.mean([np.exp(-((a - 100.0) / 40.0) ** 2) for a in leg_angles]) if leg_angles else 0.0
        hip_ids = [i for i in (11, 12) if visible[i]]
        if hip_ids and reference_box is not None:
            hip = keypoints[hip_ids].mean(axis=0)
            rx1, ry1, rx2, ry2 = reference_box
            dx = max(rx1 - hip[0], 0, hip[0] - rx2)
            dy = max(ry1 - hip[1], 0, hip[1] - ry2)
            norm = max(1.0, target_box[3] - target_box[1])
            contact = float(np.exp(-3.0 * math.hypot(dx, dy) / norm))
        else:
            contact = 0.0
        # Contact thuộc relation score; không tính lặp lại trong pose score.
        pose = 0.55 * coverage + 0.45 * float(angle_score)
    else:
        angle_score = np.mean([np.exp(-((a - 170.0) / 30.0) ** 2) for a in leg_angles]) if leg_angles else 0.0
        ankle_ids = [i for i in (15, 16) if visible[i]]
        if ankle_ids:
            ankle_y = float(keypoints[ankle_ids, 1].mean())
            norm = max(1.0, target_box[3] - target_box[1])
            contact = float(np.exp(-abs(ankle_y - target_box[3]) / (0.20 * norm)))
        else:
            contact = 0.0
        pose = 0.45 * coverage + 0.35 * float(angle_score) + 0.20 * contact
    return float(np.clip(pose, 0, 1)), coverage, contact



### 16.3. Hard constraints và soft ranking

Hard validity trả lời “candidate có dùng được không?”; soft score chỉ xếp hạng các mức chất lượng. Detection confidence chỉ tạo gate và được lưu để audit, không có trọng số trong soft score.


In [ ]:
SOFT_WEIGHTS = {
    "free_placement": {
        "placement": .25, "scale": .25, "pose": .30, "context": .20,
    },
    "object_relative": {
        "placement": .20, "scale": .15, "pose": .20,
        "relation": .15, "reference": .15, "context": .15,
    },
    "physical_interaction": {
        "placement": .10, "scale": .10, "pose": .30,
        "relation": .20, "reference": .15, "context": .15,
    },
}
assert all(abs(sum(w.values()) - 1.0) < 1e-9 for w in SOFT_WEIGHTS.values())


def relation_score(person_box, reference_box, context_type, contact_score):
    if context_type == "free_placement":
        return np.nan
    if reference_box is None:
        return 0.0
    if context_type == "physical_interaction":
        overlap = box_iou(person_box, reference_box)
        return float(np.clip(0.70 * contact_score + 0.30 * min(1.0, 5.0 * overlap), 0, 1))
    pc = (person_box[0] + person_box[2]) / 2
    rc = (reference_box[0] + reference_box[2]) / 2
    scale = max(1.0, reference_box[2] - reference_box[0])
    distance = abs(pc - rc) / scale
    separation = 1.0 - min(1.0, box_iou(person_box, reference_box) * 3.0)
    return float(np.clip(0.65 * np.exp(-distance) + 0.35 * separation, 0, 1))


def weighted_soft_score(values, weights):
    available = {k: v for k, v in values.items() if k in weights and np.isfinite(v)}
    denominator = sum(weights[k] for k in available)
    if denominator == 0:
        return 0.0
    return float(sum(weights[k] * v for k, v in available.items()) / denominator)


manifest_lookup = dataset_df.set_index("sample_id").to_dict("index")
score_rows = []

if RUN_EVALUATION:
    for run in runs_df.itertuples(index=False):
        if run.status != "success":
            score_rows.append({
                "sample_id": run.sample_id,
                "spec_id": run.spec_id,
                "heap_rank": run.heap_rank,
                "selection_score": run.selection_score,
                "context_type": run.context_type,
                "seed": run.seed,
                "output_path": run.output_path,
                "scoring_version": CONFIG["scoring_version"],
                "hard_valid": 0,
                "failure_reasons": "generation_failed",
                "person_count": 0,
                "detection_confidence": 0.0,
                "person_box": "null",
                "placement_score": 0.0,
                "scale_score": 0.0,
                "reference_scale_score": np.nan,
                "pose_score": 0.0,
                "pose_coverage": 0.0,
                "contact_score": 0.0,
                "relation_score": 0.0,
                "reference_preservation_score": np.nan,
                "context_preservation_score": np.nan,
                "soft_score": 0.0,
            })
            continue
        meta = manifest_lookup[run.sample_id]
        image = Image.open(run.output_path).convert("RGB")
        original_native = Image.open(INPUT_DIR / meta["background_path"]).convert("RGB")
        old_size = original_native.size
        original = original_native.resize(image.size, Image.Resampling.LANCZOS)
        hard_mask = np.asarray(
            Image.open(INPUT_DIR / meta["mask_path"]).convert("L").resize(image.size, Image.Resampling.NEAREST)
        ) > 127
        semantic = np.asarray(
            Image.open(INPUT_DIR / meta["semantic_mask_path"]).convert("L").resize(image.size, Image.Resampling.NEAREST)
        )
        target_box = resize_box(parse_json(meta["target_box"]), old_size, image.size)
        reference_box = resize_box(parse_json(meta["reference_box"]), old_size, image.size)
        reference_range = parse_json(meta["reference_scale_range"])

        people = segment_people(image)
        selected = select_target_person(people, target_box)
        person_count = len(people)
        if selected is None:
            person_box, person_mask, detection_confidence = None, np.zeros_like(hard_mask), 0.0
            placement = scale = reference_scale = pose = pose_coverage = contact = 0.0
            relation = 0.0
        else:
            person_box = selected["box"]
            person_mask = selected["mask"]
            detection_confidence = selected["score"]
            placement = placement_score(person_box, target_box)
            scale, reference_scale = scale_score(
                person_box, target_box, reference_box, reference_range
            )
            keypoints, keypoint_scores = estimate_pose(image, person_box)
            pose, pose_coverage, contact = pose_metrics(
                keypoints, keypoint_scores, run.context_type, reference_box, target_box
            )
            relation = relation_score(person_box, reference_box, run.context_type, contact)

        person_dilated = ndimage.binary_dilation(person_mask, iterations=5)
        object_ids = parse_json(meta["object_label_ids"], [])
        reference_mask = np.isin(semantic, object_ids) if object_ids else np.zeros_like(hard_mask)
        reference_visible = hard_mask & reference_mask & ~person_dilated
        reference_preservation = preservation_score(
            original, image, reference_visible
        )
        context_region = hard_mask & ~person_dilated & ~ndimage.binary_dilation(reference_mask, iterations=3)
        context_preservation = preservation_score(
            original, image, context_region
        )

        values = {
            "placement": placement,
            "scale": scale,
            "pose": pose,
            "relation": relation,
            "reference": reference_preservation,
            "context": context_preservation,
        }
        soft_score = weighted_soft_score(values, SOFT_WEIGHTS[run.context_type])

        failures = []
        if selected is None:
            failures.append("no_person")
        else:
            if person_count != 1:
                failures.append("person_count")
            if placement < CONFIG["hard_min_placement"]:
                failures.append("placement")
            if scale < CONFIG["hard_min_scale"]:
                failures.append("scale")
            if pose < CONFIG["hard_min_pose"]:
                failures.append("pose")
            if run.context_type != "free_placement" and relation < CONFIG["hard_min_relation"]:
                failures.append("relation")
            if np.isfinite(reference_preservation) and reference_preservation < CONFIG["hard_min_reference_preservation"]:
                failures.append("reference_changed")
            if np.isfinite(context_preservation) and context_preservation < CONFIG["hard_min_context_preservation"]:
                failures.append("context_changed")

        hard_valid = int(not failures)
        score_rows.append({
            "sample_id": run.sample_id,
            "spec_id": run.spec_id,
            "heap_rank": run.heap_rank,
            "selection_score": run.selection_score,
            "context_type": run.context_type,
            "seed": run.seed,
            "output_path": run.output_path,
            "scoring_version": CONFIG["scoring_version"],
            "hard_valid": hard_valid,
            "failure_reasons": "|".join(failures),
            "person_count": person_count,
            "detection_confidence": detection_confidence,
            "person_box": json.dumps(person_box),
            "placement_score": placement,
            "scale_score": scale,
            "reference_scale_score": reference_scale,
            "pose_score": pose,
            "pose_coverage": pose_coverage,
            "contact_score": contact,
            "relation_score": relation,
            "reference_preservation_score": reference_preservation,
            "context_preservation_score": context_preservation,
            "soft_score": soft_score,
        })

    evaluation_df = pd.DataFrame(score_rows)
else:
    ranking_path = OUTPUT_DIR / "ranking.csv"
    if not ranking_path.exists():
        raise FileNotFoundError("RUN_EVALUATION=False nhưng chưa có ranking.csv")
    evaluation_df = pd.read_csv(ranking_path, sep=";", encoding="utf-8-sig")
    versions = set(evaluation_df.get("scoring_version", pd.Series(dtype=str)).dropna())
    if versions != {CONFIG["scoring_version"]}:
        raise RuntimeError("ranking.csv dùng scorer cũ; hãy đặt RUN_EVALUATION=True để chấm lại.")
    evaluation_df = evaluation_df.drop(columns=["auto_rank"], errors="ignore")

ranking_df = evaluation_df.sort_values(
    ["sample_id", "hard_valid", "soft_score"], ascending=[True, False, False]
)
ranking_df["auto_rank"] = ranking_df.groupby("sample_id").cumcount() + 1
ranking_df.to_csv(OUTPUT_DIR / "ranking.csv", index=False, sep=";", encoding="utf-8-sig")
display(ranking_df.head())


## 17. Within-sample và between-sample variance

`generation_score` được suy ra tạm thời từ `hard_valid` và `soft_score`, không lưu lặp trong `ranking.csv`. Within-sample variance phản ánh latent seed; between-sample variance phản ánh chênh lệch mean giữa background trong cùng specification. Đây là decomposition mô tả, chưa phải causal estimate.


In [ ]:
def build_budget_table(frame, budgets):
    rows = []
    for sample_id, group in frame.groupby("sample_id"):
        group = group.sort_values("seed")
        meta = group.iloc[0]
        for budget in budgets:
            prefix = group.head(budget)
            valid = prefix[prefix["hard_valid"] == 1]
            rows.append({
                "sample_id": sample_id,
                "spec_id": meta["spec_id"],
                "context_type": meta["context_type"],
                "budget": budget,
                "success": int(len(valid) > 0),
                "num_valid": len(valid),
                "best_score": float(valid["soft_score"].max()) if len(valid) else 0.0,
            })
    return pd.DataFrame(rows)


analysis_df = evaluation_df.assign(
    generation_score=evaluation_df["soft_score"].where(
        evaluation_df["hard_valid"].eq(1), 0.0
    )
)
budget_df = build_budget_table(analysis_df, CONFIG["best_of_budgets"])
budget_features = budget_df.pivot(
    index="sample_id", columns="budget", values="best_score"
).rename(columns=lambda budget: f"best_of_{budget}_score").reset_index()

sample_stats = analysis_df.groupby(
    ["sample_id", "spec_id", "context_type", "heap_rank", "selection_score"]
).agg(
    num_seeds=("seed", "count"),
    auto_success_rate=("hard_valid", "mean"),
    mean_generation_score=("generation_score", "mean"),
    within_sample_variance=("generation_score", "var"),
).reset_index().merge(budget_features, on="sample_id")
sample_stats["selection_score_percentile"] = sample_stats.groupby("spec_id")[
    "selection_score"
].rank(method="average", pct=True)

variance_rows = []
for (spec_id, context_type), group in sample_stats.groupby(["spec_id", "context_type"]):
    within = float(group["within_sample_variance"].mean())
    raw_between = float(group["mean_generation_score"].var(ddof=1))
    mean_seed_count = float(group["num_seeds"].mean())
    between = max(0.0, raw_between - within / max(1.0, mean_seed_count))
    total = within + between
    variance_rows.append({
        "spec_id": spec_id,
        "context_type": context_type,
        "num_backgrounds": len(group),
        "mean_num_seeds": mean_seed_count,
        "mean_within_sample_variance": within,
        "raw_variance_of_sample_means": raw_between,
        "estimated_between_sample_variance": between,
        "between_variance_fraction": between / total if total > 0 else np.nan,
    })

variance_df = pd.DataFrame(variance_rows)
variance_df.to_csv(OUTPUT_DIR / "variance_decomposition.csv", index=False, sep=";", encoding="utf-8-sig")
sample_stats.to_csv(OUTPUT_DIR / "summary_by_sample.csv", index=False, sep=";", encoding="utf-8-sig")
context_stats = analysis_df.groupby("context_type").agg(
    num_samples=("sample_id", "nunique"),
    num_candidates=("seed", "count"),
    auto_success_rate=("hard_valid", "mean"),
    mean_generation_score=("generation_score", "mean"),
    mean_soft_score=("soft_score", "mean"),
).reset_index()
context_stats.to_csv(OUTPUT_DIR / "summary_by_context.csv", index=False, sep=";", encoding="utf-8-sig")
display(variance_df)
display(context_stats)


## 18. Candidate-selection score có dự đoán generation success không?

Notebook báo cả Pearson và Spearman giữa heap selection score với automatic success rate, mean generation score và Best-of-1/4/8/16. Tương quan với human labels được tính riêng trong manual validation notebook.


In [ ]:
def correlation_rows(frame, x, outcomes, source, group="all"):
    rows = []
    for outcome in outcomes:
        valid = frame[[x, outcome]].dropna()
        if len(valid) < 3 or valid[x].nunique() < 2 or valid[outcome].nunique() < 2:
            pearson = spearman = np.nan
            pearson_p = spearman_p = np.nan
        else:
            pearson, pearson_p = stats.pearsonr(valid[x], valid[outcome])
            spearman, spearman_p = stats.spearmanr(valid[x], valid[outcome])
        rows.append({
            "source": source, "group": group,
            "predictor": x, "outcome": outcome, "n": len(valid),
            "pearson_r": pearson, "pearson_p": pearson_p,
            "spearman_rho": spearman, "spearman_p": spearman_p,
        })
    return rows


correlation_outcomes = [
    "auto_success_rate", "mean_generation_score",
    "best_of_1_score", "best_of_4_score", "best_of_8_score", "best_of_16_score",
]
automatic_correlation_rows = correlation_rows(
    sample_stats, "selection_score", correlation_outcomes, "automatic"
)
automatic_correlation_rows.extend(correlation_rows(
    sample_stats,
    "selection_score_percentile",
    correlation_outcomes,
    "automatic_within_spec_normalized",
))
for context_type, group in sample_stats.groupby("context_type"):
    automatic_correlation_rows.extend(correlation_rows(
        group, "selection_score", correlation_outcomes,
        "automatic", group=context_type,
    ))
correlation_df = pd.DataFrame(automatic_correlation_rows)
correlation_df.to_csv(
    OUTPUT_DIR / "selection_score_correlations.csv",
    index=False, sep=";", encoding="utf-8-sig",
)
display(correlation_df)


## 19. Best-of-1/4/8/16

Các budget dùng cùng thứ tự seed và là các prefix lồng nhau, nhờ đó có thể so sánh lợi ích khi tăng N mà không thay đổi các seed đã quan sát trước đó. Adaptive-budget simulation được hoãn khỏi v1 baseline.


In [ ]:
budget_summary = budget_df.groupby(["context_type", "budget"]).agg(
    num_samples=("sample_id", "nunique"),
    success_rate=("success", "mean"),
    mean_best_score=("best_score", "mean"),
    std_best_score=("best_score", "std"),
).reset_index()
budget_df.to_csv(OUTPUT_DIR / "best_of_n_by_sample.csv", index=False, sep=";", encoding="utf-8-sig")
budget_summary.to_csv(OUTPUT_DIR / "best_of_n_summary.csv", index=False, sep=";", encoding="utf-8-sig")


display(budget_summary)


## 20. Lưu experiment manifest

Manifest ghi lại thiết kế background × seed, evaluator, hard thresholds và Best-of-N budgets để các lần chạy có thể được đối chiếu nhất quán.


In [ ]:
experiment_manifest = {
    **CONFIG,
    "created_at": datetime.now().isoformat(),
    "design": "specification x background x latent_seed",
    "num_specifications": int(dataset_df["spec_id"].nunique()),
    "num_background_samples": int(dataset_df["sample_id"].nunique()),
    "num_seeds": len(seeds),
    "num_generated_candidates": int(len(runs_df)),
    "dataset_manifest_path": str(DATASET_MANIFEST_PATH),
    "candidate_audit_path": (
        str(CANDIDATE_AUDIT_PATH) if CANDIDATE_AUDIT_PATH.exists() else None
    ),
    "hard_constraints": [
        "single_person", "placement", "scale", "pose", "relation",
        "reference_preservation", "context_preservation",
    ],
    "soft_components": sorted({name for weights in SOFT_WEIGHTS.values() for name in weights}),
}
manifest_path = OUTPUT_DIR / "experiment_manifest.json"
manifest_path.write_text(
    json.dumps(experiment_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved:", manifest_path)
